### Name: Blessing Adeniji
### Degree: MSc Artifical Intelligence Online
### Capstone Project: AI-Generated Text Detection - Deepfakes
Purpose: to verify dataset structure and justify the 512-token limit

In [ ]:
import pandas as pd

# Load train RAID splits
raid = pd.read_csv("data_splits/RAID_train.csv")

# print results
print("Columns:", raid.columns.tolist())
print("\nShape:", raid.shape)

# If there's a domain column, count unique domains
for col in raid.columns:
    if raid[col].dtype == object and raid[col].nunique() < 50:
        print(f"\n{col}: {raid[col].nunique()} unique values")
        print(raid[col].unique())

Columns: ['text', 'label']

Shape: (205429, 2)


In [ ]:
# to confirm the size and columns from the preprocessing technique
for name in ["MAGE_train", "ChatGPT-Research-Abstracts_train", "GPT-Wiki-intro_train"]:
    df = pd.read_csv(f"data_splits/{name}.csv")
    print(name, "| shape:", df.shape, "| columns:", df.columns.tolist())

MAGE_train | shape: (130645, 3) | columns: ['text', 'label', 'src']
ChatGPT-Research-Abstracts_train | shape: (14000, 2) | columns: ['text', 'label']
GPT-Wiki-intro_train | shape: (210000, 2) | columns: ['text', 'label']


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Use the tokenizer 
tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m")

datasets = {
    "RAID": "data_splits/RAID_train.csv",
    "MAGE": "data_splits/MAGE_train.csv",
    "ChatGPT Abstracts": "data_splits/ChatGPT-Research-Abstracts_train.csv",
    "GPT-Wiki-Intro": "data_splits/GPT-Wiki-intro_train.csv",
}

# Measures true token length per dataset without truncation and justify the length 512-tokens used during training
for name, path in datasets.items():
    df = pd.read_csv(path)
    # Count tokens per text (no truncation, to see the true length)
    lengths = df["text"].apply(lambda t: len(tokenizer(str(t), truncation=False)["input_ids"]))
    under_512 = (lengths <= 512).mean() * 100
    print(f"{name}: {under_512:.1f}% under 512 tokens "
          f"| median {int(lengths.median())}, "
          f"mean {int(lengths.mean())}, "
          f"max {int(lengths.max())}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9838 > 8192). Running this sequence through the model will result in indexing errors


RAID: 73.5% under 512 tokens | median 362, mean 660, max 127276
MAGE: 86.4% under 512 tokens | median 148, mean 257, max 12536
ChatGPT Abstracts: 96.6% under 512 tokens | median 243, mean 258, max 935
GPT-Wiki-Intro: 99.6% under 512 tokens | median 229, mean 227, max 1552
